In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow import keras
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

`TextVectorization` layer implements the STIE process i.e. Standardize, Tokenize, Index and Encode. So we configure this layer by telling what STIE we want exactly.
1. Default standardization used to remove punctuation and converting everything into lowercase.
2. Default tokenization is done such that `split='whitespace'`
3. Set output mode to `multi_hot`.

In [3]:
text_vectorization = keras.layers.TextVectorization(
    output_mode='multi_hot',
    standardize='lower_and_strip_punctuation',
    split='whitespace'
)

Now that the layer has been configured, we have to run it on a bunch of text to "populate" it. We use `adapt()` method for it.

In [4]:
dataset=["I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
    "Hola! are you laid-back in Mexico"]

In [5]:
text_vectorization.adapt(dataset)

In [6]:
vocabulary = text_vectorization.get_vocabulary()

In [7]:
len(vocabulary)
print(pd.DataFrame(vocabulary))

           0
0      [UNK]
1      erase
2        you
3      write
4       then
5    rewrite
6      poppy
7     mexico
8   laidback
9         in
10         i
11      hola
12    blooms
13       are
14       and
15     again
16         a


Here, `[UNK]` is assigned to such word which is not present in dataset.

In [8]:
test_sentense = "I write, rewrite and still rewrite again"
encoded_sentense = text_vectorization(test_sentense)
print(encoded_sentense)

tf.Tensor([1 0 0 1 0 1 0 0 0 0 1 0 0 0 1 1 0], shape=(17,), dtype=int64)


In [9]:
"still" in vocabulary

False

In [10]:
text_vectorization("Sloan, HODL, DMD")

<tf.Tensor: shape=(17,), dtype=int64, numpy=array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])>

In [11]:
# Read data from URL

train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url,index_col=0)
val_df = pd.read_csv(val_url,index_col=0)
test_df = pd.read_csv(test_url,index_col=0)


print(f"""
Train samples: {train_df.shape[0]}
Validation samples: {val_df.shape[0]}
Test samples: {test_df.shape[0]}
""")


Train samples: 48991
Validation samples: 16331
Test samples: 21774



In [12]:
train_df.head()

,Lyric,Genre
0,"Oh, girl. I can't get ready (Can't get ready f...",Pop
1,We met on a rainy evening in the summertime. D...,Pop
2,We carried you in our arms. On Independence Da...,Rock
3,I know he loved you. A long time ago. I ain't ...,Pop
4,Paralysis through analysis. Yellow moral uncle...,Rock


In [13]:
train_df.tail()

,Lyric,Genre
48986,"[Hook]. Beamer, Benz, Or Bentley. Beamer, Benz...",Hip Hop
48987,You never listen to me. I know I'm better off ...,Pop
48988,Things have come to a pretty pass. Our romance...,Pop
48989,"Little baby, on my shoulder. I could fall into...",Pop
48990,Music : Rudolf Schenker. Lyrics: Klaus Meine. ...,Rock


In [14]:
# Checking the proportion of each genre in the training dataset
train_df['Genre'].value_counts() / train_df.shape[0]

,count
Genre,
Rock,0.549448
Pop,0.295136
Hip Hop,0.155416


In [15]:
# One hot encoding
y_train = pd.get_dummies(train_df['Genre']).to_numpy(dtype=np.int64)
y_test = pd.get_dummies(test_df['Genre']).to_numpy(dtype=np.int64)
y_val = pd.get_dummies(val_df['Genre']).to_numpy(dtype=np.int64)

In [16]:
y_train

array([[0, 1, 0],
       [0, 1, 0],
       [0, 0, 1],
       ...,
       [0, 1, 0],
       [0, 1, 0],
       [0, 0, 1]])

# Baseline Model (Bag of words)
1. We will tokenize at word level, and each token will be a one word (unigram)
2. We will use the one-hot encoding that converts each token into a binary vector indicating teh presence of token. Note that when we tokenize at word level and use a one-hot encoding to indicate the presence of the token (or as we will see later, the count of the word), the model is called bag of words.
3. `max_tokens` defines the size of of the vocabulary layer is allowed to construct. If the number of token in the `adapt` dataset exceeds this number then the layer will choose teh `max_tokens` most frequent tokens and ignore the rest.

In [17]:
max_tokens = 5000
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='multi_hot'
)

In [18]:
# STIE Process(Standardize, Tokenize, Indexing, Encoding)

text_vectorization.adapt(train_df['Lyric'])

In [19]:
# lets look at 20 most and 20 least common words in our vectorization
text_vectorization.get_vocabulary()[:20]

['[UNK]',
 np.str_('the'),
 np.str_('you'),
 np.str_('i'),
 np.str_('to'),
 np.str_('and'),
 np.str_('a'),
 np.str_('me'),
 np.str_('it'),
 np.str_('my'),
 np.str_('in'),
 np.str_('im'),
 np.str_('on'),
 np.str_('your'),
 np.str_('that'),
 np.str_('of'),
 np.str_('all'),
 np.str_('be'),
 np.str_('is'),
 np.str_('we')]

In [20]:
text_vectorization.get_vocabulary()[-20:]

[np.str_('eden'),
 np.str_('dagger'),
 np.str_('curve'),
 np.str_('cheddar'),
 np.str_('brew'),
 np.str_('appears'),
 np.str_('vacant'),
 np.str_('universal'),
 np.str_('unholy'),
 np.str_('terrified'),
 np.str_('stickin'),
 np.str_('rumble'),
 np.str_('rug'),
 np.str_('pam'),
 np.str_('os'),
 np.str_('ooohh'),
 np.str_('motto'),
 np.str_('marshall'),
 np.str_('loyalty'),
 np.str_('legacy')]

In [21]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

Each phrase is now a vector of dimention 5000

In [22]:
X_train

<tf.Tensor: shape=(48991, 5000), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])>

In [23]:
inputs = keras.Input(shape=(max_tokens, ))
x = keras.layers.Dense(8, activation="relu")(inputs)
outputs = keras.layers.Dense(3, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │        40,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,035 (156.39 KB)

 Trainable params: 40,035 (156.39 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [25]:
model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7311 - loss: 0.6167 - val_accuracy: 0.7529 - val_loss: 0.5701
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7736 - loss: 0.5205 - val_accuracy: 0.7497 - val_loss: 0.5778
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7906 - loss: 0.4835 - val_accuracy: 0.7469 - val_loss: 0.5976
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8066 - loss: 0.4536 - val_accuracy: 0.7451 - val_loss: 0.6182
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8191 - loss: 0.4285 - val_accuracy: 0.7427 - val_loss: 0.6466
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8295 - loss: 0.4072 - val_accuracy: 0.7409 - val_loss: 0.6700
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8383 - loss: 0.3878 - val_accuracy: 0.7368 - val_loss: 0.7024
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8465 - loss: 0.3697 

In [26]:
model.evaluate(X_test, y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7208 - loss: 0.8239


[0.8238804936408997, 0.7207679152488708]

# Adding Context with bigrams

E.g. "the cat sat on the mat"
Monograms: {cat, mat, sat, on, the}
Bigrams: {cat, mat, sat, on, the, cat mat, cat sat, cat on, the cat, the mat, mat sat, on mat, on sat, the sat, on the}

In [27]:
# text_vactorization = keras.layers.TextVectorization(
#     ngrams=2,
#     output_mode="multi_hot"
# )

In [28]:
# text_vectorization.adapt(["the cat sat on the mat."])

In [29]:
# text_vectorization.get_vocabulary()

In [30]:
text_vectorization = keras.layers.TextVectorization(
    ngrams=2,
    max_tokens=10000, # increased from 5k to 20 to accomodate bigrams
    output_mode='multi_hot'
)

In [31]:
text_vectorization.adapt(train_df['Lyric'])

In [32]:
text_vectorization.get_vocabulary()[:10]

['[UNK]',
 np.str_('the'),
 np.str_('you'),
 np.str_('i'),
 np.str_('to'),
 np.str_('and'),
 np.str_('a'),
 np.str_('me'),
 np.str_('it'),
 np.str_('my')]

In [33]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [38]:
inputs = keras.Input(shape=(10000, ))
x = keras.layers.Dense(8, activation="relu")(inputs)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │        80,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,035 (312.64 KB)

 Trainable params: 80,035 (312.64 KB)

 Non-trainable params: 0 (0.00 B)

# Regularization Strategy: Dropout
Randomly zero out the output from some of the nodes (typically 50% of the nodes) in a hidden layer (implemented as a "dropout layer" in keras)

In [39]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [40]:
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=3, batch_size=32)

Epoch 1/3
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.6512 - loss: 0.7426 - val_accuracy: 0.7348 - val_loss: 0.6094
Epoch 2/3
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7048 - loss: 0.6487 - val_accuracy: 0.7464 - val_loss: 0.5889
Epoch 3/3
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7326 - loss: 0.5993 - val_accuracy: 0.7492 - val_loss: 0.5877


In [41]:
model.evaluate(X_test, y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7406 - loss: 0.6003


[0.600265622138977, 0.7405621409416199]